# Task 5: Claim verification + error analysis

Retrieval gives a ranked list of abstracts per claim. Here each of the top abstracts gets a verdict: does it **SUPPORT** the claim, **CONTRADICT** it, or say nothing decisive (no row, and a claim with no rows at all is NOT ENOUGH INFO).

Steps:
1. pick the best retriever (highest val nDCG@10 of the seven runs)
2. fine-tune an NLI classifier on the training claims (the zero-shot version is kept as a baseline)
3. tune the decision rule on val, then predict val and eval
4. tag every wrong val claim with an error category

Needs the zip from notebook 4. GPU runtime (T4), about 10-15 min.

In [2]:
# upload NLP_Inter_IIT.zip (the one notebook 4 downloaded in its last cell) and unpack it
# on colab this opens the file picker. If the folder is already here (rerun, or running locally) nothing happens.
import zipfile
from pathlib import Path

def find_root():
    # folder that contains data/, wherever the notebook was opened from
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "NLP_Inter_IIT"]:
        if (p / "data" / "corpus.jsonl").exists():
            return p

if find_root() is None:
    zips = list(Path.cwd().glob("*.zip"))        # zip already dragged into the colab file panel
    if not zips:
        from google.colab import files
        zips = [Path(name) for name in files.upload()]
    newest = max(zips, key=lambda z: z.stat().st_mtime)
    zipfile.ZipFile(newest).extractall(".")
    print("unpacked", newest.name)

ROOT = find_root()
assert ROOT is not None, "data/corpus.jsonl not found, upload NLP_Inter_IIT.zip"
assert (ROOT / "work" / "scores_ft_hard_val.npy").exists(), "this zip has no notebook 4 outputs: upload the zip that notebook 4 downloaded in its last cell"
print("project folder:", ROOT)
print(sorted(p.name for p in (ROOT / "predictions").iterdir()))

Saving NLP_Inter_IIT_5.zip to NLP_Inter_IIT_5.zip
unpacked NLP_Inter_IIT_5.zip
project folder: /content/NLP_Inter_IIT
['t1_bm25.trec', 't2_general.trec', 't2_scientific.trec', 't3_hybrid.trec', 't3_rerank.trec', 't4_hard.trec', 't4_random.trec', 'val_bm25.trec', 'val_general.trec']


In [1]:
%pip install -q transformers sentencepiece protobuf

In [3]:
import csv, json, math, random, re, sys, time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DATA, PRED, WORK = ROOT / "data", ROOT / "predictions", ROOT / "work"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "none (cpu)")

cuda | Tesla T4


## Data and helpers

In [4]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = sorted(read_jsonl(DATA / "corpus.jsonl"), key=lambda d: d["doc_id"])
doc_ids = np.array([d["doc_id"] for d in corpus])
doc_text = {d["doc_id"]: d["title"] + " " + " ".join(d["abstract"]) for d in corpus}
doc_title = {d["doc_id"]: d["title"] for d in corpus}
claims = {s: sorted(read_jsonl(DATA / f"{s}_claims.jsonl"), key=lambda c: c["id"]) for s in ["train", "val", "eval"]}
claim_text = {s: {c["id"]: c["claim"] for c in claims[s]} for s in claims}

TOP_K = 100

def gold_labels(split):
    # claim id -> {doc_id: "SUPPORT" / "CONTRADICT"}  (empty dict = NOT ENOUGH INFO)
    return {c["id"]: {e["doc_id"]: e["label"] for e in c["evidence"]} for c in claims[split]}

def make_run(scores, split, k=TOP_K):
    order = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    return {c["id"]: [(int(doc_ids[j]), float(scores[i, j])) for j in order[i]] for i, c in enumerate(claims[split])}

def ndcg10(run, split):
    vals = []
    for cid, g in gold_labels(split).items():
        if not g:
            continue
        ranked = [d for d, _ in run[cid]][:10]
        dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(ranked) if d in g)
        vals.append(dcg / sum(1 / math.log2(r + 2) for r in range(min(len(g), 10))))
    return round(float(np.mean(vals)), 4)

results_log = json.load(open(WORK / "results_log.json"))

## Best retriever

All seven runs side by side on val. The best one supplies the candidate abstracts for the classifier.

In [5]:
MATRIX_SYSTEMS = {"bm25": "bm25", "general": "general", "scientific": "scientific", "hybrid": "hybrid",
                  "ft_random": "ft_random", "ft_hard": "ft_hard"}

def load_run(system, split):
    if system == "rerank":
        if split == "train":
            # the reranker was only run on val/eval. For training claims the first-stage ranking it reranks is used.
            system = results_log["task3"]["reranked_system"]
        else:
            raw = json.load(open(WORK / f"run_rerank_{split}.json"))
            return {int(cid): [(int(d), float(s)) for d, s in rows] for cid, rows in raw.items()}
    return make_run(np.load(WORK / f"scores_{MATRIX_SYSTEMS[system]}_{split}.npy"), split)

val_table = pd.Series({name: ndcg10(load_run(name, "val"), "val")
                       for name in ["bm25", "general", "scientific", "hybrid", "rerank", "ft_random", "ft_hard"]},
                      name="val_ndcg10")
print(val_table.to_string())

best_retriever = val_table.idxmax()
print("\nbest retriever:", best_retriever)
runs = {split: load_run(best_retriever, split) for split in ["train", "val", "eval"]}

bm25          0.8735
general       0.9328
scientific    0.8509
hybrid        0.9352
rerank        0.9259
ft_random     0.9153
ft_hard       0.9460

best retriever: ft_hard


## Candidates and training examples

The classifier looks at the top `POOL` abstracts of each claim (how many of those are actually used is tuned later).

Training examples from the 647 training claims:
- **(claim, gold abstract)** with its SUPPORT / CONTRADICT label
- **(claim, retrieved abstract that is not gold)** labelled neutral. These come from the top of the same retriever, because that is exactly what the classifier has to reject at prediction time: abstracts on the right topic that don't settle the claim. Claims with no evidence contribute neutral examples only.

In [6]:
POOL = 10            # abstracts per claim that get a verdict
N_NEUTRAL = 3        # neutral training examples per training claim

pools = {split: {cid: [d for d, _ in ranked[:POOL]] for cid, ranked in runs[split].items()} for split in ["val", "eval"]}

train_gold = gold_labels("train")
train_examples = []          # (claim, abstract text, label)
for c in claims["train"]:
    for did, label in train_gold[c["id"]].items():
        train_examples.append((c["claim"], doc_text[did], label))
    non_gold = [d for d, _ in runs["train"][c["id"]][:POOL] if d not in train_gold[c["id"]]]
    for did in non_gold[:N_NEUTRAL]:
        train_examples.append((c["claim"], doc_text[did], "NEUTRAL"))

print(len(train_examples), "training examples:", dict(Counter(lab for _, _, lab in train_examples)))

# how often is the gold abstract among the candidates at all? (upper bound for the classifier's recall)
val_gold = gold_labels("val")
n_gold = sum(len(g) for g in val_gold.values())
for m in [1, 3, 5, 10]:
    found = sum(1 for cid, g in val_gold.items() for d in g if d in pools["val"][cid][:m])
    print(f"gold abstracts inside the top {m:2d}: {found}/{n_gold} = {found / n_gold:.3f}")

2381 training examples: {'NEUTRAL': 1941, 'CONTRADICT': 150, 'SUPPORT': 290}
gold abstracts inside the top  1: 94/124 = 0.758
gold abstracts inside the top  3: 115/124 = 0.927
gold abstracts inside the top  5: 119/124 = 0.960
gold abstracts inside the top 10: 122/124 = 0.984


## Classifier

An NLI model takes a *premise* and a *hypothesis* and predicts entailment / neutral / contradiction. Here the premise is the abstract and the hypothesis is the claim, so

- entailment → SUPPORT
- contradiction → CONTRADICT
- neutral → no row

The starting point is DeBERTa-v3-base already trained on MNLI, FEVER and ANLI (184M parameters). FEVER is itself a claim verification dataset, so the zero-shot model is a meaningful baseline. It is then fine-tuned on our training examples.

In [16]:
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
MAX_LEN = 512
EPOCHS = 5
BATCH_SIZE = 16
LR = 2e-5
USE_AMP = True       # mixed precision. If the loss turns into nan, set this to False and BATCH_SIZE to 8.

tok = AutoTokenizer.from_pretrained(NLI_MODEL)
# .float(): this checkpoint is stored in half precision and would otherwise be loaded that way,
# and half-precision weights can't be trained with the gradient scaler
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).float().to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params | {next(model.parameters()).dtype} | labels: {model.config.id2label}")
assert sum(p.numel() for p in model.parameters()) <= 400e6

# positions of the three classes in this model's output
name_to_id = {name.lower(): i for i, name in model.config.id2label.items()}
ENT, CON, NEU = name_to_id["entailment"], name_to_id["contradiction"], name_to_id["neutral"]
LABEL_ID = {"SUPPORT": ENT, "CONTRADICT": CON, "NEUTRAL": NEU}
amp = USE_AMP and device == "cuda"

def tokenize(abstracts, claim_list):
    # premise = abstract, hypothesis = claim. If the pair is too long only the abstract is cut.
    return tok(abstracts, claim_list, padding=True, truncation="only_first", max_length=MAX_LEN, return_tensors="pt").to(device)

@torch.no_grad()
def predict(split):
    # probabilities for every (claim, candidate abstract): {claim_id: array (POOL, 3)} with columns support / contradict / neutral
    model.eval()
    pairs = [(cid, did) for cid, dids in pools[split].items() for did in dids]
    order = np.argsort([-len(doc_text[did]) for _, did in pairs])        # longest first, less padding per batch
    probs = np.zeros((len(pairs), 3), dtype=np.float32)
    for i in range(0, len(pairs), 32):
        idx = order[i:i + 32]
        batch = tokenize([doc_text[pairs[j][1]] for j in idx], [claim_text[split][pairs[j][0]] for j in idx])
        with torch.autocast(device_type=device, enabled=amp):
            logits = model(**batch).logits
        probs[idx] = torch.softmax(logits.float(), dim=-1)[:, [ENT, CON, NEU]].cpu().numpy()
    return {cid: probs[i * POOL:(i + 1) * POOL] for i, cid in enumerate(pools[split])}

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

184M params | torch.float32 | labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


## Decision rule and metric

For one claim: look at its top `m` candidates, keep those where the SUPPORT or CONTRADICT probability is at least `tau`, and keep at most the 3 most confident (file format limit).

One dataset-specific rule: in the training data no claim has both a supporting and a contradicting abstract, so all rows of a claim get the label of its most confident row.

`m` and `tau` are tuned on val to maximise F1 over (claim, abstract, label) triples, which is the official metric.

In [17]:
def decide(probs, split, m, tau):
    # -> {claim_id: [(doc_id, label), ...]}  (empty list = NOT ENOUGH INFO)
    out = {}
    for cid, p in probs.items():
        rows = []
        for j, did in enumerate(pools[split][cid][:m]):
            conf = max(p[j, 0], p[j, 1])
            if conf >= tau:
                rows.append((float(conf), did, "SUPPORT" if p[j, 0] >= p[j, 1] else "CONTRADICT"))
        rows.sort(key=lambda r: -r[0])
        rows = rows[:3]
        out[cid] = [(did, rows[0][2]) for _, did, _ in rows]
    return out

def prf(pred, gold):
    # precision / recall / F1 over (claim, doc, label) triples
    p = {(cid, did, lab) for cid, rows in pred.items() for did, lab in rows}
    g = {(cid, did, lab) for cid, docs in gold.items() for did, lab in docs.items()}
    tp = len(p & g)
    precision = tp / len(p) if p else 0.0
    recall = tp / len(g) if g else 0.0
    f1 = 2 * precision * recall / (precision + recall) if tp else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}

M_GRID = [1, 2, 3, 5, 10]
TAU_GRID = [round(0.30 + 0.05 * i, 2) for i in range(14)]        # 0.30, 0.35, ..., 0.95

def tune(probs_val):
    best = None
    for m in M_GRID:
        for tau in TAU_GRID:
            score = prf(decide(probs_val, "val", m, tau), val_gold)
            if best is None or score["f1"] > best["f1"]:
                best = {"m": m, "tau": tau, **score}
    return best

In [18]:
# zero-shot baseline: the pretrained NLI model without any training on our claims
zero_shot = tune(predict("val"))
print("zero-shot on val:", zero_shot)

zero-shot on val: {'m': 2, 'tau': 0.55, 'precision': 0.5896, 'recall': 0.6371, 'f1': 0.6124}


## Fine-tuning

In [19]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_examples) / BATCH_SIZE)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * EPOCHS * steps_per_epoch), EPOCHS * steps_per_epoch)
scaler = torch.amp.GradScaler(device, enabled=amp)
rng = random.Random(SEED)

history = [{"epoch": 0, "train_loss": None, **zero_shot}]
best = {"epoch": 0, **zero_shot}
best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

for epoch in range(1, EPOCHS + 1):
    model.train()
    order = list(range(len(train_examples)))
    rng.shuffle(order)
    losses = []
    for start in tqdm(range(0, len(order), BATCH_SIZE), desc=f"epoch {epoch}", leave=False):
        batch = [train_examples[i] for i in order[start:start + BATCH_SIZE]]
        enc = tokenize([abstract for _, abstract, _ in batch], [claim for claim, _, _ in batch])
        labels = torch.tensor([LABEL_ID[lab] for _, _, lab in batch], device=device)
        with torch.autocast(device_type=device, enabled=amp):
            logits = model(**enc).logits
        loss = F.cross_entropy(logits.float(), labels)
        assert not torch.isnan(loss), "loss is nan: set USE_AMP = False and BATCH_SIZE = 8 in the settings cell and rerun from there"
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        losses.append(loss.item())

    row = {"epoch": epoch, "train_loss": round(float(np.mean(losses)), 4), **tune(predict("val"))}
    history.append(row)
    print(row)
    if row["f1"] > best["f1"]:                       # keep the weights of the best epoch (val F1)
        best = dict(row)
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

pd.DataFrame(history)

epoch 1:   0%|          | 0/149 [00:00<?, ?it/s]

/tmp/ipykernel_628/2654210669.py:28: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


{'epoch': 1, 'train_loss': 0.4364, 'm': 2, 'tau': 0.3, 'precision': 0.75, 'recall': 0.7016, 'f1': 0.725}


epoch 2:   0%|          | 0/149 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.2257, 'm': 2, 'tau': 0.3, 'precision': 0.7739, 'recall': 0.7177, 'f1': 0.7448}


...

epoch 4:   0%|          | 0/149 [00:00<?, ?it/s]

{'epoch': 4, 'train_loss': 0.0495, 'm': 2, 'tau': 0.35, 'precision': 0.7823, 'recall': 0.7823, 'f1': 0.7823}


epoch 5:   0%|          | 0/149 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.0278, 'm': 3, 'tau': 0.3, 'precision': 0.7778, 'recall': 0.7903, 'f1': 0.784}


,epoch,train_loss,m,tau,precision,recall,f1
0,0,NaN,2,0.55,0.5896,0.6371,0.6124
1,1,0.4364,2,0.30,0.7500,0.7016,0.7250
2,2,0.2257,2,0.30,0.7739,0.7177,0.7448
3,3,0.1304,2,0.30,0.7845,0.7339,0.7583
4,4,0.0495,2,0.35,0.7823,0.7823,0.7823
5,5,0.0278,3,0.30,0.7778,0.7903,0.7840


In [20]:
# final model = best epoch on val (epoch 0 would mean the zero-shot model was not beaten)
model.load_state_dict(best_state)
mode = "zero-shot" if best["epoch"] == 0 else "finetuned"
M, TAU = best["m"], best["tau"]
print(f"using epoch {best['epoch']} ({mode}), top {M} candidates, threshold {TAU}")

pred_val = decide(predict("val"), "val", M, TAU)
pred_eval = decide(predict("eval"), "eval", M, TAU)

val_scores = prf(pred_val, val_gold)
print("val:", val_scores)
print("claims with at least one row: val", sum(bool(r) for r in pred_val.values()), "of", len(pred_val),
      "| eval", sum(bool(r) for r in pred_eval.values()), "of", len(pred_eval))

using epoch 5 (finetuned), top 3 candidates, threshold 0.3
val: {'precision': 0.7778, 'recall': 0.7903, 'f1': 0.784}
claims with at least one row: val 99 of 162 | eval 157 of 300


In [21]:
def write_verification(path, pred):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["claim_id", "doc_id", "label"])
        for cid in sorted(pred):
            for did, label in pred[cid]:
                w.writerow([cid, did, label])

write_verification(PRED / "t5_verification.csv", pred_eval)
write_verification(PRED / "t5_val_verification.csv", pred_val)

## Error analysis on val

A val claim counts as an error when its predicted (abstract, label) pairs are not exactly the gold pairs. Each error gets one category, checked in this order:

| category | rule |
|---|---|
| `retrieval_miss` | a gold abstract was not among the candidates the classifier looked at |
| `negation` | a gold abstract was found but given the opposite label, or the claim contains a negation and was misjudged |
| `needs_multiple_docs` | the claim has several gold abstracts and only some of them were predicted |
| `numerical` | the claim contains a number, percentage or quantity |
| `entity_mismatch` | a label was given to an abstract that is not gold (related topic, different entity or setting) |
| `other` | none of the above (typically: gold abstract was seen but judged neutral) |

These are heuristics. The first rule is exact, the rest are a best guess at the cause, so it is worth reading through the examples below.

In [22]:
NEGATION = re.compile(r"\b(not|no|never|none|neither|nor|without|lack|lacks|lacking|absent|absence|fail|fails|failed|cannot|unable|unaffected|unrelated|independent|insufficient)\b", re.I)
NUMBER = re.compile(r"\d|%|\b(percent|fold|half|twice|double|doubles|triple|majority|minority|two|three|four|five|six|seven|eight|nine|ten|hundred|thousand|million)\b", re.I)

def categorize(cid):
    gold = val_gold[cid]
    pred = dict(pred_val[cid])
    seen = set(pools["val"][cid][:M])
    if any(did not in seen for did in gold):
        return "retrieval_miss"
    if any(did in pred and pred[did] != label for did, label in gold.items()):
        return "negation"
    if len(gold) >= 2 and any(pred.get(did) == label for did, label in gold.items()):
        return "needs_multiple_docs"
    claim = claim_text["val"][cid]
    if NUMBER.search(claim):
        return "numerical"
    if NEGATION.search(claim):
        return "negation"
    if any(did not in gold for did in pred):
        return "entity_mismatch"
    return "other"

errors = {cid: categorize(cid) for cid in sorted(val_gold) if set(pred_val[cid]) != set(val_gold[cid].items())}

with open(PRED / "t5_val_errors.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["claim_id", "category"])
    for cid, cat in errors.items():
        w.writerow([cid, cat])

counts = pd.Series(Counter(errors.values())).sort_values(ascending=False)
miss_fraction = round(float(counts.get("retrieval_miss", 0)) / max(len(errors), 1), 4)
print(f"{len(errors)} of {len(val_gold)} val claims are errors")
print(counts.to_string())
print(f"\nretrieval_miss share of errors (report Q4): {counts.get('retrieval_miss', 0)}/{len(errors)} = {miss_fraction}")

45 of 162 val claims are errors
numerical              15
other                   9
retrieval_miss          8
entity_mismatch         7
needs_multiple_docs     3
negation                3

retrieval_miss share of errors (report Q4): 8/45 = 0.1778


In [23]:
# a few examples per category, to check that the tags make sense
def show(cid):
    print(f"[{cid}] {claim_text['val'][cid]}")
    print("    gold     :", [(doc_title[d][:60], lab) for d, lab in val_gold[cid].items()] or "NOT ENOUGH INFO")
    print("    predicted:", [(doc_title[d][:60], lab) for d, lab in pred_val[cid]] or "NOT ENOUGH INFO")

for cat in counts.index:
    print(f"\n=== {cat} ({counts[cat]}) ===")
    for cid in [c for c, k in errors.items() if k == cat][:3]:
        show(cid)


=== numerical (15) ===
[24] 90% of patients with melanoma and an objective response to PD-1 blockade will experience a progression in their melanoma.
    gold     : NOT ENOUGH INFO
    predicted: [('Association of Pembrolizumab With Tumor Response and Surviva', 'CONTRADICT')]
[64] ATM and Rad3 related protein have no role in sensing DNA damage.
    gold     : NOT ENOUGH INFO
    predicted: [('Molecular mechanisms of mammalian DNA repair and the DNA dam', 'CONTRADICT')]
[189] Broadly HIV-1 Neutralizing Antibodies (bnAb) 10EB have no affinity for phospholipids.
    gold     : [('Broad and potent neutralization of HIV-1 by a gp41-specific ', 'SUPPORT')]
    predicted: NOT ENOUGH INFO

=== other (9) ===
[34] A deficiency of folate increases blood levels of homocysteine.
    gold     : [('Randomized trial of folic acid supplementation and serum hom', 'SUPPORT')]
    predicted: NOT ENOUGH INFO
[235] Cell autonomous sex determination in somatic cells occurs in Galliformes.
    gold     : [('

In [24]:
# official format check for the three files
sys.path.append(str(ROOT))
from check_format import check_verification, check_errors

all_docs = set(doc_ids.tolist())
val_ids = {c["id"] for c in claims["val"]}
check_verification(PRED / "t5_verification.csv", {c["id"] for c in claims["eval"]}, all_docs)
check_verification(PRED / "t5_val_verification.csv", val_ids, all_docs)
check_errors(PRED / "t5_val_errors.csv", val_ids)
print("t5_verification.csv, t5_val_verification.csv and t5_val_errors.csv pass check_format")

t5_verification.csv, t5_val_verification.csv and t5_val_errors.csv pass check_format


In [25]:
retriever_desc = {"hybrid": "hybrid: " + results_log["task3"]["hybrid_method"],
                  "rerank": f"rerank: {results_log['task3']['reranker_model']} on top {results_log['task3']['rerank_top_k']} of {results_log['task3']['reranked_system']}",
                  "general": "general: " + results_log["task2"]["general_model"],
                  "scientific": "scientific: " + results_log["task2"]["scientific_model"],
                  "bm25": "bm25",
                  "ft_random": "fine-tuned (random negatives): " + results_log["task4"]["base_model"],
                  "ft_hard": "fine-tuned (hard negatives): " + results_log["task4"]["base_model"]}[best_retriever]

results_log["task5"] = {
    "retriever_used": retriever_desc,
    "nli_model": NLI_MODEL,
    "zero_shot_or_finetuned": mode,
    "val_f1": val_scores["f1"],
    "val_precision": val_scores["precision"],
    "val_recall": val_scores["recall"],
    "zero_shot_val_f1": zero_shot["f1"],
    "best_epoch": best["epoch"],
    "candidates_per_claim": M,
    "threshold": TAU,
    "history": history,
    "val_errors": int(len(errors)),
    "error_counts": {k: int(v) for k, v in counts.items()},
    "retrieval_miss_fraction": miss_fraction,
    "all_runs_val_ndcg10": val_table.to_dict(),
}
json.dump(results_log, open(WORK / "results_log.json", "w"), indent=2)
{k: results_log["task5"][k] for k in ["retriever_used", "nli_model", "zero_shot_or_finetuned", "val_f1", "zero_shot_val_f1", "retrieval_miss_fraction"]}

{'retriever_used': 'fine-tuned (hard negatives): BAAI/bge-large-en-v1.5',
 'nli_model': 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli',
 'zero_shot_or_finetuned': 'finetuned',
 'val_f1': 0.784,
 'zero_shot_val_f1': 0.6124,
 'retrieval_miss_fraction': 0.1778}

## Save for the next notebook

Packs data, predictions and work into `NLP_Inter_IIT.zip` and downloads it (colab only). Upload this zip in the first cell of the next notebook.

In [26]:
if "google.colab" in sys.modules:
    zip_path = Path.cwd() / "NLP_Inter_IIT_6.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for item in ["data", "predictions", "work", "check_format.py", "results_template.json"]:
            p = ROOT / item
            if not p.exists():
                continue
            for f in ([p] if p.is_file() else sorted(p.rglob("*"))):
                if f.is_file():
                    z.write(f, Path("NLP_Inter_IIT") / f.relative_to(ROOT))
    print(zip_path.name, f"{zip_path.stat().st_size / 1e6:.0f} MB")
    from google.colab import files
    files.download(str(zip_path))

NLP_Inter_IIT_6.zip 123 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>